# 🏷️ MSM Tagging Automation with Google Translation API


This notebook performs automated **relevancy and sentiment tagging** for MSM data.  
It uses **Google Cloud Translation API** to translate non-English text to English before analysis.


In [ ]:

# ====================================================
# Install dependencies
# ====================================================
!pip install google-cloud-translate pandas numpy openpyxl --quiet


In [ ]:

# ====================================================
# Import libraries
# ====================================================
import os
import pandas as pd
import numpy as np
import re
from google.cloud import translate_v2 as translate

# Initialize translator client
translate_client = translate.Client()

def translate_text(text, target="en"):
    """Translate text to English using Google Translate API (v2)."""
    if not text or not isinstance(text, str):
        return text
    try:
        result = translate_client.translate(text, target_language=target)
        return result.get("translatedText", text)
    except Exception:
        return text


In [ ]:

# ====================================================
# Load Excel file
# ====================================================
file_path = "UAE - MSM_Cleaned.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
print("✅ Data loaded successfully!")
print("Columns:", df.columns.tolist())
df.head()


In [ ]:

# ====================================================
# Relevancy tagging
# ====================================================
def check_relevancy(row):
    input_name = str(row.get("Input Name", "")).lower().strip()
    headline = str(row.get("Headline", ""))
    opening = str(row.get("Opening Text", ""))
    headline_en = translate_text(headline)
    opening_en = translate_text(opening)
    combined_text = (headline_en + " " + opening_en).lower()
    return "Relevant" if input_name and input_name in combined_text else "Not Relevant"

df.insert(0, "Relevancy", df.apply(check_relevancy, axis=1))
print("✅ Relevancy tagging complete!")
df[['Relevancy', 'Input Name', 'Headline', 'Opening Text']].head(10)


In [ ]:

# ====================================================
# Sentiment retagging with extended negative keywords
# ====================================================
NEGATIVE_KEYWORDS = [
    "inflation", "cautious", "debt", "lower", "loss", "down", "fell", "downgrade",
    "cuts", "reduce", "drop", "warn", "slash", "mixed", "reduction", "fed cut",
    "fed rate cut", "rate cut", "insufficient", "cut", "weak", "uncertain", "mute",
    "pressur", "fluctuate", "diverge", "subpoena", "blocks", "plummeted", "fined",
    "costly", "overweight", "pullback", "crash", "amid", "concern", "dispute"
]

POSITIVE_KEYWORDS = [
    "appointment", "promotion", "joins as", "new ceo", "hired", "partnership",
    "award", "expanding", "growth", "invest", "funding", "launched", "wins",
    "secured", "recognized", "acquisition"
]

def retag_sentiment(row):
    original = str(row.get("Sentiment", "")).strip()
    headline = translate_text(str(row.get("Headline", "")))
    opening = translate_text(str(row.get("Opening Text", "")))
    text = (headline + " " + opening).lower()

    if original.lower() == "positive":
        return "Positive"

    for kw in POSITIVE_KEYWORDS:
        if kw in text:
            return "Positive"

    for kw in NEGATIVE_KEYWORDS:
        if kw in text:
            return "Negative"

    if re.search(r"stock price (drop|decline|fall|decrease)|share price (drop|fall)", text):
        return "Neutral"

    return "Neutral"

df["New_Sentiment"] = df.apply(retag_sentiment, axis=1)
print("✅ Sentiment retagging complete!")
df[['Input Name', 'Headline', 'Sentiment', 'New_Sentiment']].head(15)


In [ ]:

# ====================================================
# Export final results
# ====================================================
output_path = "UAE_MSM_Tagged_Translated.xlsx"
df.to_excel(output_path, index=False)
print(f"🎉 Exported to: {output_path}")
print("\nSentiment Summary:")
print(df['New_Sentiment'].value_counts())
print("\nRelevancy Summary:")
print(df['Relevancy'].value_counts())
